In [64]:
import pandas as pd
from pathlib import Path

data_path = Path("../data/PS_20174392719_1491204439457_log.csv")

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Shape: (6362620, 11)

First 5 rows:


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [65]:

# 1. All column names
print("COLUMN NAMES:")
print(df.columns.tolist())

# 2. Data types and non-null counts
print("\nDATASET INFORMATION:")
df.info()

# 3. Missing values in each column
print("\nMISSING VALUES:")
print(df.isnull().sum())

# 4. Transaction types
print("\nTRANSACTION TYPES:")
print(df["type"].value_counts())

# 5. Fraud vs non-fraud counts
print("\nFRAUD LABEL COUNTS:")
print(df["isFraud"].value_counts())

# 6. Fraud percentage
print("\nFRAUD PERCENTAGE:")
print((df["isFraud"].value_counts(normalize=True) * 100).round(4))


COLUMN NAMES:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

DATASET INFORMATION:
<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB

MISSING VALUES:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

TRANSACTION TYPES:
type

In [66]:

print("Missing values:")
print(df.isnull().sum().to_string())

print("\nTransaction types:")
print(df["type"].value_counts().to_string())

print("\nFraud counts:")
print(df["isFraud"].value_counts().to_string())

print("\nFlagged fraud counts:")
print(df["isFlaggedFraud"].value_counts().to_string())

print("\nDuplicate rows:", df.duplicated().sum())


Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432

Fraud counts:
isFraud
0    6354407
1       8213

Flagged fraud counts:
isFlaggedFraud
0    6362604
1         16

Duplicate rows: 0


In [67]:

# Fraud and total transactions by transaction type
type_analysis = df.groupby("type")["isFraud"].agg(
    total_transactions="count",
    fraud_transactions="sum"
)

type_analysis["fraud_percentage"] = (
    type_analysis["fraud_transactions"]
    / type_analysis["total_transactions"] * 100
).round(4)

print(type_analysis)

# Check the time-step range
print("\nTime-step range:")
print("Minimum step:", df["step"].min())
print("Maximum step:", df["step"].max())

# Check which transaction types appear among fraud cases
print("\nFraud transactions by type:")
print(
    df.loc[df["isFraud"] == 1, "type"]
    .value_counts()
)


          total_transactions  fraud_transactions  fraud_percentage
type                                                              
CASH_IN              1399284                   0            0.0000
CASH_OUT             2237500                4116            0.1840
DEBIT                  41432                   0            0.0000
PAYMENT              2151495                   0            0.0000
TRANSFER              532909                4097            0.7688

Time-step range:
Minimum step: 1
Maximum step: 743

Fraud transactions by type:
type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64


In [68]:

# Is the dataset arranged in chronological order?
print("Rows already sorted by step:", df["step"].is_monotonic_increasing)

# Divide the timeline into five consecutive ranges
time_bins = [0, 148, 296, 444, 592, 743]
time_labels = [
    "Steps 1-148",
    "Steps 149-296",
    "Steps 297-444",
    "Steps 445-592",
    "Steps 593-743"
]

df["time_period"] = pd.cut(
    df["step"],
    bins=time_bins,
    labels=time_labels,
    include_lowest=True
)

# Count total and fraudulent transactions in each period
time_analysis = df.groupby(
    "time_period", observed=False
)["isFraud"].agg(
    total_transactions="count",
    fraud_transactions="sum"
)

time_analysis["fraud_percentage"] = (
    time_analysis["fraud_transactions"]
    / time_analysis["total_transactions"] * 100
).round(4)

print("\nFraud distribution across time:")
print(time_analysis)


Rows already sorted by step: True

Fraud distribution across time:
               total_transactions  fraud_transactions  fraud_percentage
time_period                                                            
Steps 1-148               1511332                1674            0.1108
Steps 149-296             2449014                1697            0.0693
Steps 297-444             2050527                1630            0.0795
Steps 445-592              221627                1544            0.6967
Steps 593-743              130120                1668            1.2819


In [69]:

# Inspect the numerical features
numeric_cols = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

print(df[numeric_cols].describe().round(2).to_string())

# Inspect transaction types
print("\nTransaction type categories:")
print(sorted(df["type"].unique()))

# Check how often the existing fraud flag is set
print("\nFraud label vs existing flag:")
print(pd.crosstab(df["isFraud"], df["isFlaggedFraud"]))


             step       amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  newbalanceDest
count  6362620.00   6362620.00     6362620.00      6362620.00    6.362620e+06    6.362620e+06
mean       243.40    179861.90      833883.10       855113.67    1.100702e+06    1.224996e+06
std        142.33    603858.23     2888242.67      2924048.50    3.399180e+06    3.674129e+06
min          1.00         0.00           0.00            0.00    0.000000e+00    0.000000e+00
25%        156.00     13389.57           0.00            0.00    0.000000e+00    0.000000e+00
50%        239.00     74871.94       14208.00            0.00    1.327057e+05    2.146614e+05
75%        335.00    208721.48      107315.18       144258.41    9.430367e+05    1.111909e+06
max        743.00  92445516.64    59585040.37     49585040.37    3.560159e+08    3.561793e+08

Transaction type categories:
['CASH_IN', 'CASH_OUT', 'DEBIT', 'PAYMENT', 'TRANSFER']

Fraud label vs existing flag:
isFlaggedFraud        0   1
isFraud  

In [70]:

# Check whether fraud transactions have unusual balance patterns

fraud_rows = df[df["isFraud"] == 1]
normal_rows = df[df["isFraud"] == 0]

print("Fraud transaction types:")
print(fraud_rows["type"].value_counts())

print("\nAverage amount:")
print("Fraud:", round(fraud_rows["amount"].mean(), 2))
print("Normal:", round(normal_rows["amount"].mean(), 2))

print("\nAverage sender balance before transaction:")
print("Fraud:", round(fraud_rows["oldbalanceOrg"].mean(), 2))
print("Normal:", round(normal_rows["oldbalanceOrg"].mean(), 2))


Fraud transaction types:
type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64

Average amount:
Fraud: 1467967.3
Normal: 178197.04

Average sender balance before transaction:
Fraud: 1649667.61
Normal: 832828.71


In [71]:
# Target: what the model will predict
y = df["isFraud"]

# Input features for our first traditional ML model
feature_cols = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

X = df[feature_cols].copy()

print("Input features shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

Input features shape: (6362620, 7)
Target shape: (6362620,)

Target distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [72]:
from sklearn.model_selection import train_test_split

# 1. Random split: 80% training, 20% testing
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("RANDOM SPLIT")
print("Training shape:", X_train_random.shape)
print("Testing shape:", X_test_random.shape)

print("\nFraud cases in training:", y_train_random.sum())
print("Fraud cases in testing:", y_test_random.sum())

RANDOM SPLIT
Training shape: (5090096, 7)
Testing shape: (1272524, 7)

Fraud cases in training: 6570
Fraud cases in testing: 1643


In [73]:
# Check the chronological time range
print("Minimum step:", df["step"].min())
print("Maximum step:", df["step"].max())

print("\nTransactions per step:")
print(df["step"].value_counts().sort_index().head())

# Find the boundary near 80% of all transactions
split_index = int(len(df) * 0.8)
split_step = df.iloc[split_index]["step"]

print("\n80% row boundary index:", split_index)
print("Step at the boundary:", split_step)

Minimum step: 1
Maximum step: 743

Transactions per step:
step
1    2708
2    1014
3     552
4     565
5     665
Name: count, dtype: int64

80% row boundary index: 5090096
Step at the boundary: 355


In [74]:
# Temporal split: past for training, later data for testing
train_mask = df["step"] < 355
test_mask = df["step"] >= 355

X_train_temporal = X.loc[train_mask].copy()
X_test_temporal = X.loc[test_mask].copy()

y_train_temporal = y.loc[train_mask].copy()
y_test_temporal = y.loc[test_mask].copy()

print("TEMPORAL SPLIT")

print("\nTraining rows:", len(X_train_temporal))
print("Testing rows:", len(X_test_temporal))

print("\nTraining step range:",
      df.loc[train_mask, "step"].min(),
      "to",
      df.loc[train_mask, "step"].max())

print("Testing step range:",
      df.loc[test_mask, "step"].min(),
      "to",
      df.loc[test_mask, "step"].max())

print("\nFraud cases in training:", y_train_temporal.sum())
print("Fraud cases in testing:", y_test_temporal.sum())

TEMPORAL SPLIT

Training rows: 5069097
Testing rows: 1293523

Training step range: 1 to 354
Testing step range: 355 to 743

Fraud cases in training: 3955
Fraud cases in testing: 4258


In [75]:
# Compare fraud rates across both temporal sets

'''step ko feature rakhna sahi hai ya nahi. Temporal generalization mein time feature valid ho sakta hai.
lekin changing fraud rates ki wajah se model time pattern par bahut zyada depend bhi kar sakta hai.'''

print("Training fraud rate (%):",
      round(y_train_temporal.mean() * 100, 4))

print("Testing fraud rate (%):",
      round(y_test_temporal.mean() * 100, 4))

print("\nTraining transaction types:")
print(X_train_temporal["type"].value_counts())

print("\nTesting transaction types:")
print(X_test_temporal["type"].value_counts())

Training fraud rate (%): 0.078
Testing fraud rate (%): 0.3292

Training transaction types:
type
CASH_OUT    1796898
PAYMENT     1706555
CASH_IN     1112415
TRANSFER     421007
DEBIT         32222
Name: count, dtype: int64

Testing transaction types:
type
PAYMENT     444940
CASH_OUT    440602
CASH_IN     286869
TRANSFER    111902
DEBIT         9210
Name: count, dtype: int64


In [76]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

print("Preprocessing and evaluation tools imported successfully!")

Preprocessing and evaluation tools imported successfully!


In [77]:
# Separate numerical and categorical columns
numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

categorical_features = ["type"]

# Preprocessing rules
preprocessor_random = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Fit only on training data, then transform both sets
X_train_random_processed = preprocessor_random.fit_transform(X_train_random)
X_test_random_processed = preprocessor_random.transform(X_test_random)

print("Random split preprocessing complete!")
print("Processed training shape:", X_train_random_processed.shape)
print("Processed testing shape:", X_test_random_processed.shape)

Random split preprocessing complete!
Processed training shape: (5090096, 11)
Processed testing shape: (1272524, 11)


In [78]:
# Create the Logistic Regression model for random split
lr_random = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train the model
lr_random.fit(X_train_random_processed, y_train_random)

print("Logistic Regression training complete!")

Logistic Regression training complete!


In [79]:
# Predict on unseen test data
y_pred_random = lr_random.predict(X_test_random_processed)

# Get fraud probabilities for PR-AUC and ROC-AUC
y_prob_random = lr_random.predict_proba(X_test_random_processed)[:, 1]

# Calculate evaluation metrics
precision = precision_score(y_test_random, y_pred_random, zero_division=0)
recall = recall_score(y_test_random, y_pred_random, zero_division=0)
f1 = f1_score(y_test_random, y_pred_random, zero_division=0)
pr_auc = average_precision_score(y_test_random, y_prob_random)
roc_auc = roc_auc_score(y_test_random, y_prob_random)

print("LOGISTIC REGRESSION — RANDOM SPLIT")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))
print("PR-AUC (Average Precision):", round(pr_auc, 4))
print("ROC-AUC:", round(roc_auc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_random, y_pred_random))

LOGISTIC REGRESSION — RANDOM SPLIT
Precision: 0.0238
Recall: 0.9598
F1-score: 0.0464
PR-AUC (Average Precision): 0.5817
ROC-AUC: 0.9908

Confusion Matrix:
[[1206133   64748]
 [     66    1577]]


In [80]:
# Create a separate preprocessor for temporal data
preprocessor_temporal = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Fit only on temporal training data
X_train_temporal_processed = preprocessor_temporal.fit_transform(
    X_train_temporal
)

# Transform future-period test data
X_test_temporal_processed = preprocessor_temporal.transform(
    X_test_temporal
)

print("Temporal preprocessing complete!")
print("Training shape:", X_train_temporal_processed.shape)
print("Testing shape:", X_test_temporal_processed.shape)

Temporal preprocessing complete!
Training shape: (5069097, 11)
Testing shape: (1293523, 11)


In [81]:
# Create Logistic Regression for temporal split
lr_temporal = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train only on earlier transactions
lr_temporal.fit(
    X_train_temporal_processed,
    y_train_temporal
)

print("Temporal Logistic Regression training complete!")

Temporal Logistic Regression training complete!


In [82]:
# Predict on future-period test data
y_pred_temporal = lr_temporal.predict(
    X_test_temporal_processed
)

# Get fraud probabilities
y_prob_temporal = lr_temporal.predict_proba(
    X_test_temporal_processed
)[:, 1]

# Calculate evaluation metrics
precision_temporal = precision_score(
    y_test_temporal, y_pred_temporal, zero_division=0
)

recall_temporal = recall_score(
    y_test_temporal, y_pred_temporal, zero_division=0
)

f1_temporal = f1_score(
    y_test_temporal, y_pred_temporal, zero_division=0
)

pr_auc_temporal = average_precision_score(
    y_test_temporal, y_prob_temporal
)

roc_auc_temporal = roc_auc_score(
    y_test_temporal, y_prob_temporal
)

print("LOGISTIC REGRESSION — TEMPORAL SPLIT")
print("Precision:", round(precision_temporal, 4))
print("Recall:", round(recall_temporal, 4))
print("F1-score:", round(f1_temporal, 4))
print("PR-AUC (Average Precision):", round(pr_auc_temporal, 4))
print("ROC-AUC:", round(roc_auc_temporal, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_temporal, y_pred_temporal))

LOGISTIC REGRESSION — TEMPORAL SPLIT
Precision: 0.1102
Recall: 0.8828
F1-score: 0.1959
PR-AUC (Average Precision): 0.6801
ROC-AUC: 0.988

Confusion Matrix:
[[1258906   30359]
 [    499    3759]]


In [83]:
import pandas as pd

lr_results = pd.DataFrame([
    {
        "Experiment": "LR-RANDOM-01",
        "Model": "Logistic Regression",
        "Split": "Random",
        "Precision": 0.0238,
        "Recall": 0.9598,
        "F1-score": 0.0464,
        "PR-AUC": 0.5817,
        "ROC-AUC": 0.9908
    },
    {
        "Experiment": "LR-TEMPORAL-01",
        "Model": "Logistic Regression",
        "Split": "Temporal",
        "Precision": 0.1102,
        "Recall": 0.8828,
        "F1-score": 0.1959,
        "PR-AUC": 0.6801,
        "ROC-AUC": 0.9880
    }
])

lr_results

,Experiment,Model,Split,Precision,Recall,F1-score,PR-AUC,ROC-AUC
0,LR-RANDOM-01,Logistic Regression,Random,0.0238,0.9598,0.0464,0.5817,0.9908
1,LR-TEMPORAL-01,Logistic Regression,Temporal,0.1102,0.8828,0.1959,0.6801,0.9880


In [84]:
from pathlib import Path

results_folder = Path("../results")
results_folder.mkdir(parents=True, exist_ok=True)

lr_results.to_csv(
    results_folder / "logistic_regression_results.csv",
    index=False
)

print("Results saved successfully!")

Results saved successfully!


In [85]:
print(lr_results)
print("\nSaved file:", (results_folder / "logistic_regression_results.csv").resolve())

       Experiment                Model     Split  Precision  Recall  F1-score  \
0    LR-RANDOM-01  Logistic Regression    Random     0.0238  0.9598    0.0464   
1  LR-TEMPORAL-01  Logistic Regression  Temporal     0.1102  0.8828    0.1959   

   PR-AUC  ROC-AUC  
0  0.5817   0.9908  
1  0.6801   0.9880  

Saved file: C:\Users\PC\Documents\Major Project\results\logistic_regression_results.csv


In [86]:
from sklearn.ensemble import RandomForestClassifier

print("Random Forest imported successfully!")

Random Forest imported successfully!


In [87]:
rf_random = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,          #Defining the settings of Raandom Forest model
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created successfully!")

rf_random.fit(X_train_random_processed, y_train_random)

print("Random Forest training completed!")

Random Forest model created successfully!
Random Forest training completed!


In [88]:
# Predict fraud labels and probabilities
y_pred_rf_random = rf_random.predict(X_test_random_processed)

y_prob_rf_random = rf_random.predict_proba(
    X_test_random_processed
)[:, 1]

# Calculate evaluation metrics
precision_rf_random = precision_score(
    y_test_random, y_pred_rf_random, zero_division=0
)
recall_rf_random = recall_score(
    y_test_random, y_pred_rf_random, zero_division=0
)
f1_rf_random = f1_score(
    y_test_random, y_pred_rf_random, zero_division=0
)
pr_auc_rf_random = average_precision_score(
    y_test_random, y_prob_rf_random
)
roc_auc_rf_random = roc_auc_score(
    y_test_random, y_prob_rf_random
)

print("RANDOM FOREST — RANDOM SPLIT")
print("Precision:", round(precision_rf_random, 4))
print("Recall:", round(recall_rf_random, 4))
print("F1-score:", round(f1_rf_random, 4))
print("PR-AUC (Average Precision):", round(pr_auc_rf_random, 4))
print("ROC-AUC:", round(roc_auc_rf_random, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_random, y_pred_rf_random))

RANDOM FOREST — RANDOM SPLIT
Precision: 0.0983
Recall: 0.9897
F1-score: 0.1789
PR-AUC (Average Precision): 0.9193
ROC-AUC: 0.9994

Confusion Matrix:
[[1255971   14910]
 [     17    1626]]


In [89]:
rf_temporal = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_temporal.fit(
    X_train_temporal_processed,
    y_train_temporal
)

print("Temporal Random Forest training completed!")

Temporal Random Forest training completed!


In [90]:
y_pred_rf_temporal = rf_temporal.predict(
    X_test_temporal_processed
)

y_prob_rf_temporal = rf_temporal.predict_proba(
    X_test_temporal_processed
)[:, 1]

precision_rf_temporal = precision_score(
    y_test_temporal, y_pred_rf_temporal, zero_division=0
)

recall_rf_temporal = recall_score(
    y_test_temporal, y_pred_rf_temporal, zero_division=0
)

f1_rf_temporal = f1_score(
    y_test_temporal, y_pred_rf_temporal, zero_division=0
)

pr_auc_rf_temporal = average_precision_score(
    y_test_temporal, y_prob_rf_temporal
)

roc_auc_rf_temporal = roc_auc_score(
    y_test_temporal, y_prob_rf_temporal
)

print("RANDOM FOREST — CORRECT TEMPORAL SPLIT")
print("Precision:", round(precision_rf_temporal, 4))
print("Recall:", round(recall_rf_temporal, 4))
print("F1-score:", round(f1_rf_temporal, 4))
print("PR-AUC (Average Precision):", round(pr_auc_rf_temporal, 4))
print("ROC-AUC:", round(roc_auc_rf_temporal, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_temporal, y_pred_rf_temporal))

RANDOM FOREST — CORRECT TEMPORAL SPLIT
Precision: 0.2796
Recall: 0.9547
F1-score: 0.4325
PR-AUC (Average Precision): 0.8995
ROC-AUC: 0.9991

Confusion Matrix:
[[1278791   10474]
 [    193    4065]]


In [91]:
rf_results = pd.DataFrame([
    {
        "Experiment": "RF-RANDOM-01",
        "Model": "Random Forest",
        "Split": "Random",
        "Precision": 0.0983,
        "Recall": 0.9897,
        "F1-score": 0.1789,
        "PR-AUC": 0.9193,
        "ROC-AUC": 0.9994
    },
    {
        "Experiment": "RF-TEMPORAL-01",
        "Model": "Random Forest",
        "Split": "Temporal",
        "Precision": 0.2796,
        "Recall": 0.9547,
        "F1-score": 0.4325,
        "PR-AUC": 0.8995,
        "ROC-AUC": 0.9991
    }
])

all_results = pd.concat(
    [lr_results, rf_results],
    ignore_index=True
)

all_results.to_csv(
    results_folder / "model_comparison_results.csv",
    index=False
)

print(all_results)
print("\nComparison results saved successfully!")

       Experiment                Model     Split  Precision  Recall  F1-score  \
0    LR-RANDOM-01  Logistic Regression    Random     0.0238  0.9598    0.0464   
1  LR-TEMPORAL-01  Logistic Regression  Temporal     0.1102  0.8828    0.1959   
2    RF-RANDOM-01        Random Forest    Random     0.0983  0.9897    0.1789   
3  RF-TEMPORAL-01        Random Forest  Temporal     0.2796  0.9547    0.4325   

   PR-AUC  ROC-AUC  
0  0.5817   0.9908  
1  0.6801   0.9880  
2  0.9193   0.9994  
3  0.8995   0.9991  

Comparison results saved successfully!


In [92]:
rf_results.to_csv(
    results_folder / "random_forest_results.csv",
    index=False
)

print("Random Forest results saved successfully!")

Random Forest results saved successfully!


In [93]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [94]:
xgb_random = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("XGBoost model created successfully!")

XGBoost model created successfully!


In [95]:
xgb_random.fit(
    X_train_random_processed,
    y_train_random
)

print("XGBoost random-split training completed!")

XGBoost random-split training completed!


In [96]:
# Predict labels and fraud probabilities
y_pred_xgb_random = xgb_random.predict(
    X_test_random_processed
)

y_prob_xgb_random = xgb_random.predict_proba(
    X_test_random_processed
)[:, 1]

# Calculate metrics
precision_xgb_random = precision_score(
    y_test_random, y_pred_xgb_random, zero_division=0
)

recall_xgb_random = recall_score(
    y_test_random, y_pred_xgb_random, zero_division=0
)

f1_xgb_random = f1_score(
    y_test_random, y_pred_xgb_random, zero_division=0
)

pr_auc_xgb_random = average_precision_score(
    y_test_random, y_prob_xgb_random
)

roc_auc_xgb_random = roc_auc_score(
    y_test_random, y_prob_xgb_random
)

print("XGBOOST — RANDOM SPLIT")
print("Precision:", round(precision_xgb_random, 4))
print("Recall:", round(recall_xgb_random, 4))
print("F1-score:", round(f1_xgb_random, 4))
print("PR-AUC (Average Precision):", round(pr_auc_xgb_random, 4))
print("ROC-AUC:", round(roc_auc_xgb_random, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_random, y_pred_xgb_random))

XGBOOST — RANDOM SPLIT
Precision: 0.973
Recall: 0.8125
F1-score: 0.8856
PR-AUC (Average Precision): 0.9544
ROC-AUC: 0.9997

Confusion Matrix:
[[1270844      37]
 [    308    1335]]


In [97]:
xgb_temporal = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("Temporal XGBoost model created successfully!")

Temporal XGBoost model created successfully!


In [98]:

xgb_temporal.fit(
    X_train_temporal_processed,
    y_train_temporal
)

print("XGBoost temporal-split training completed!")


XGBoost temporal-split training completed!


In [99]:

# Make predictions on temporal test data
y_pred_xgb_temporal = xgb_temporal.predict(X_test_temporal_processed)

# Get fraud probabilities
y_prob_xgb_temporal = xgb_temporal.predict_proba(
    X_test_temporal_processed
)[:, 1]

# Calculate evaluation metrics
precision_xgb_temporal = precision_score(
    y_test_temporal, y_pred_xgb_temporal, zero_division=0
)
recall_xgb_temporal = recall_score(
    y_test_temporal, y_pred_xgb_temporal, zero_division=0
)
f1_xgb_temporal = f1_score(
    y_test_temporal, y_pred_xgb_temporal, zero_division=0
)
pr_auc_xgb_temporal = average_precision_score(
    y_test_temporal, y_prob_xgb_temporal
)
roc_auc_xgb_temporal = roc_auc_score(
    y_test_temporal, y_prob_xgb_temporal
)

# Display results
print("XGBOOST — TEMPORAL SPLIT")
print(f"Precision: {precision_xgb_temporal:.4f}")
print(f"Recall: {recall_xgb_temporal:.4f}")
print(f"F1-score: {f1_xgb_temporal:.4f}")
print(f"PR-AUC (Average Precision): {pr_auc_xgb_temporal:.4f}")
print(f"ROC-AUC: {roc_auc_xgb_temporal:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_temporal, y_pred_xgb_temporal))


XGBOOST — TEMPORAL SPLIT
Precision: 0.9997
Recall: 0.7158
F1-score: 0.8343
PR-AUC (Average Precision): 0.9570
ROC-AUC: 0.9998

Confusion Matrix:
[[1289264       1]
 [   1210    3048]]


In [100]:

# Store XGBoost experiment results
xgb_results = pd.DataFrame([
    {
        "Experiment": "XGB-RANDOM-01",
        "Model": "XGBoost",
        "Split": "Random",
        "Precision": 0.9730,
        "Recall": 0.8125,
        "F1-score": 0.8856,
        "PR-AUC": 0.9544,
        "ROC-AUC": 0.9997
    },
    {
        "Experiment": "XGB-TEMPORAL-01",
        "Model": "XGBoost",
        "Split": "Temporal",
        "Precision": 0.9997,
        "Recall": 0.7158,
        "F1-score": 0.8343,
        "PR-AUC": 0.9570,
        "ROC-AUC": 0.9998
    }
])

# Save XGBoost results separately
xgb_results.to_csv(
    results_folder / "xgboost_results.csv",
    index=False
)

# Update the combined comparison file
all_results = pd.concat(
    [lr_results, rf_results, xgb_results],
    ignore_index=True
)

all_results.to_csv(
    results_folder / "model_comparison_results.csv",
    index=False
)

print("XGBoost results saved successfully!")
print("\nCombined model comparison:")
display(all_results)


XGBoost results saved successfully!

Combined model comparison:


,Experiment,Model,Split,Precision,Recall,F1-score,PR-AUC,ROC-AUC
0,LR-RANDOM-01,Logistic Regression,Random,0.0238,0.9598,0.0464,0.5817,0.9908
1,LR-TEMPORAL-01,Logistic Regression,Temporal,0.1102,0.8828,0.1959,0.6801,0.9880
2,RF-RANDOM-01,Random Forest,Random,0.0983,0.9897,0.1789,0.9193,0.9994
3,RF-TEMPORAL-01,Random Forest,Temporal,0.2796,0.9547,0.4325,0.8995,0.9991
4,XGB-RANDOM-01,XGBoost,Random,0.9730,0.8125,0.8856,0.9544,0.9997
5,XGB-TEMPORAL-01,XGBoost,Temporal,0.9997,0.7158,0.8343,0.9570,0.9998
